**Seoul Bike Sharing Dataset 데이터셋**

서울시의 시간대별 자전거 대여 데이터를 포함하고 있으며, 날씨와 계절 요인이 자전거 대여량에 어떤 영향을 미치는지를 분석할 수 있는 시계열형 예측 데이터셋입니다.

제공된 학습용 데이터(bike_train.csv)를 이용하여 대여 자전거 대수(Rented_Bike_Count)를 예측하는 모델을 개발하고, 개발한 모델에 기반하여 평가용 데이터(bike_test.csv)에 적용하여 얻은 대여 자전거 대수 예측 값을 아래 [제출형식]에 따라 csv 파일로 생성하여 제출하시오.
- 예측 결과는 RMSE(Root Mean Squared Error) 평가지표에 따라 평가함
- 성능이 우수한 예측 모델을 구축하기 위해서는 데이터 정제, Feature Engineering, 하이퍼 파라미터(hyper parameter) 최적화, 모델 비교 등이 필요할 수 있음. 다만, 과적합에 유의하여야 함


[[제출 형식]]
- 가. CSV 파일명: result.csv(파일명에 디렉토리/폴더 지정불가
- 나. 예측 칼럼명 : pred
- 다. 제출 칼럼 개수 : pred 칼럼 1개
- 라. 평가용 데이터 개수와 예측 결과 데이터 개수 일치 : 2,716개

[[제공 데이터]]
- 데이터 목록
- bike_train.csv : 학습용 데이터, 6,044개
- bike_test.csv : 평가용 데이터, 2,716개
- 평가용 데이터는 'Rented_Bike_Count' 칼럼 미제공

In [40]:
# 라이브러리
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error as MSE
pd.set_option('display.width', 120)

# 데이터 불러오기
path = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/"
train = pd.read_csv(path + "bike_train.csv")
test = pd.read_csv(path + "bike_test.csv")
# print(train.head(3), train.shape, sep="\n")
# print(test.head(3), test.shape, sep="\n")

# 데이터 전처리
X = train.drop(columns=['Rented_Bike_Count'])
Y = train['Rented_Bike_Count']
X_all = pd.concat([X, test])
X_all['Date'] = pd.to_datetime(X_all['Date'], format="%d/%m/%Y")
X_all['Year'] = X_all['Date'].dt.year
X_all['Month'] = X_all['Date'].dt.month
X_all['Day'] = X_all['Date'].dt.day
X_all['WeekDay'] = X_all['Date'].dt.day_name('ko_KR')
X_all = X_all.drop(columns=['Date'])
# print(X_all.head(3))
cols_obj = X_all.select_dtypes(include='object').columns
for col in cols_obj:
    X_all[col] = LabelEncoder().fit_transform(X_all[col])
# print(X_all.head(3))
X_all = pd.get_dummies(X_all, drop_first=True, dtype='int')

# 데이터 분할
X = X_all.iloc[:len(X), :]
X_submission = X_all.iloc[len(X):, :]
# print(X.shape, X_submission.shape) # (6044, 16) (2716, 16)
temp = train_test_split(X, Y, test_size=0.3, random_state=1234)
x_train, x_test, y_train, y_test = temp
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape) # (4230, 16) (1814, 16) (4230,) (1814,)

# 파이프라인 모델사전
models = {
    "Linear": Pipeline([
        ('scaler', MinMaxScaler()), ('model', LinearRegression())
    ]),
    "DecisiionTree": Pipeline([
        ('model', DecisionTreeRegressor(max_depth=10, random_state=1234))
    ]),
    "RandomForestRegressor": Pipeline([
        ('model', RandomForestRegressor(max_depth=12, random_state=1234))
    ]),
    "AdaBoost": Pipeline([
        ('model', AdaBoostRegressor(n_estimators=200, random_state=1234))
    ]),
    "GradientBoosting": Pipeline([
        ('model', GradientBoostingRegressor(n_estimators=200, random_state=1234))
    ])
}

# 성능평가 함수
def get_scores(model, x_train, x_test, y_train, y_test):
    model.fit(x_train, y_train)
    y_pred1 = abs(model.predict(x_train))
    y_pred2 = abs(model.predict(x_test))
    RMSE_train = np.sqrt(MSE(y_train, y_pred1) ** 0.5)
    RMSE_test = np.sqrt(MSE(y_test, y_pred2) ** 0.5)
    return model, RMSE_train, RMSE_test

# 모델별 성능평가
results = []
for name, model in models.items():
    model, RMSE_train, RMSE_test = get_scores(model, x_train, x_test, y_train, y_test)
    results.append({
        "Model": name, "RMSE_train": round(RMSE_train, 3), "RMSE_test": round(RMSE_test, 3)
    })
res = pd.DataFrame(results).sort_values("RMSE_test", ascending=True).reset_index(drop=True)
# print(res)

# 모델선택, 종속변수 예측
model = models[res.loc[0, "Model"]]
y_pred = model.predict(X_submission)
# print(y_pred)

# 제출파일 생성
pd.DataFrame({'pred': y_pred}).to_csv("result_type2_03_2th.csv", index=False)

# 결과확인
temp = pd.read_csv("result_type2_03_2th.csv")
print(temp['pred'].describe())
print("=" * 35)
print(Y[:len(X_submission)].describe())


count    2716.000000
mean      701.421961
std       596.018070
min         0.000000
25%       220.563109
50%       520.307051
75%      1011.090589
max      2759.755237
Name: pred, dtype: float64
count    2716.000000
mean      693.877761
std       636.814224
min         0.000000
25%       193.000000
50%       490.500000
75%      1034.250000
max      3556.000000
Name: Rented_Bike_Count, dtype: float64
